# 25 — Long-horizon survival: interrupt, resume, compact

At real horizons, interruption is normal: the session ends, the budget runs out, the process dies. A deep agent survives because everything that matters lives in **artifacts** — the plan, the files, the checkpoint — and none of it lives only in the transcript.

| survives the kill | dies with the kill |
| --- | --- |
| todo list (checkpointed) | the context window |
| virtual filesystem (checkpointed) | the tool-loop trace |
| goal + checkpoint record | anything only "remembered" |

Two rules follow. **Resume rebuilds the window from artifacts, never from the dead transcript.** And the policy must be **resume-safe**: fetch evidence from artifacts when the trace lacks it, never assume an earlier observation is still around — the synthesize branch below reads the digest files back instead of quoting past sub-agent replies.

Honest scope note: `CheckpointStore` is in-memory, so the "kill" here is the loop stopping early inside one kernel — a stand-in for process death with a durable store, which is the production swap. Within-run survival uses the third technique from Anthropic's context-engineering post: **compaction** (cell 3).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'notebooks'))
from lab_core.deep import DeepAgent, SubAgent, VirtualFS, compose_system_prompt
from lab_core.patterns import Budget, CheckpointStore

fs = VirtualFS()
fs.write('/sources/doc-a.md', 'CLAIM: deterministic spines reduce variance.\n' + 'method detail. ' * 60)
fs.write('/sources/doc-b.md', 'CLAIM: objective verification beats self-approval.\n' + 'study detail. ' * 60)

def read_claims(task, context, files):
    text = files.read(task.split()[-1])
    context.add('tool:read_file', text[:120])
    return {'summary': '; '.join(l for l in text.splitlines() if l.startswith('CLAIM:')), 'detail': text}

def policy(goal, todos, trace):
    if not todos.items:
        return {'tool': 'write_todos', 'input': {'add': ['digest doc-a', 'digest doc-b', 'write recommendation']}}
    pending = todos.next_pending()
    if pending is None:
        return {'final': 'Recommendation at /notes/recommendation.md'}
    index, last = todos.items.index(pending), (trace[-1] if trace else {})
    if pending.text.startswith('digest'):
        doc = pending.text.split()[-1]
        if last.get('tool') == 'task' and doc in str(last.get('input')):
            return {'tool': 'write_todos', 'input': {'done': index}}
        return {'tool': 'task', 'input': {'agent': 'reader', 'task': f'extract claims from /sources/{doc}.md'}}
    # Resume-safe synthesis: evidence comes from artifacts, not from memory of the trace.
    if last.get('tool') == 'write_file':
        return {'tool': 'write_todos', 'input': {'done': index}}
    reads = [t for t in trace if t.get('tool') == 'read_file']
    if not reads and last.get('tool') != 'ls':
        return {'tool': 'ls', 'input': {'prefix': '/subagents'}}
    listed = next(str(t['observation']) for t in reversed(trace) if t.get('tool') == 'ls').splitlines()
    unread = [p for p in listed if p not in [str(t.get('input', {}).get('path')) for t in reads]]
    if unread:
        return {'tool': 'read_file', 'input': {'path': unread[0]}}
    claims = [l for t in reads for l in str(t['observation']).splitlines() if l.startswith('CLAIM:')]
    return {'tool': 'write_file', 'input': {'path': '/notes/recommendation.md', 'text': '\n'.join(f'- {c}' for c in claims)}}

agent = DeepAgent('reviewer', system_prompt=compose_system_prompt('You review sources.'),
                  policy=policy, fs=fs, budget=Budget(max_steps=30),
                  checkpoints=CheckpointStore())
agent.subagents = {'reader': SubAgent('reader', 'Extract CLAIM lines.', read_claims, fs)}

result = agent.run('Review the sources and recommend', run_id='review-1', max_steps=3)  # the kill
print('completed:', result.completed, '| error:', result.error)
print('\nplan at the moment of death:\n' + agent.todos.render())
print('\ncheckpoint holds:', sorted(agent.checkpoints.load('review-1')))
leg1_task_obs = next(e.text for e in agent.context.entries if e.label == 'tool:task')

In [ ]:
# Resume: fresh budget, window rebuilt from artifacts - not from the dead transcript.
agent.budget = Budget(max_steps=30)
resumed = agent.resume('review-1', max_steps=12)
print('completed:', resumed.completed, 'in', len(resumed.trace), 'steps\n')
print(agent.todos.render())
print('\nleg-1 observation still in the window?', leg1_task_obs in agent.context.render())
print('window was rebuilt from:', [e.label for e in agent.context.entries
                                   if not e.label.startswith('tool:')])
print()
print(agent.fs.read('/notes/recommendation.md'))

In [ ]:
# Within-run survival: compaction folds old observations into one lossy summary.
from lab_core.deep import Context

context = Context(owner='long-runner')
context.pin('system', 'You are a long-running agent.')
for i in range(10):
    context.add('obs', f'observation {i}: ' + 'detail ' * 12)
print('before:', context.tokens(), 'tokens across', len(context.entries), 'entries')

summary = context.compact(keep_last=2)
print('after: ', context.tokens(), 'tokens across', len(context.entries), 'entries')
print('folded into:', repr(summary))
print('entries now:', [e.label for e in context.entries])
# DeepAgent(context_limit=...) triggers exactly this automatically mid-run.

**Trade-offs.** Compaction is lossy by design — which is why the order of defenses is notes and files first, compaction last: what was offloaded to an artifact survives compaction untouched. And resume *trusts* artifacts: a poisoned plan file or a tampered note is an injection vector that walks straight into the rebuilt window, so consequential actions after a resume deserve the same approval gates as notebook 05.